In [1]:
import json
from pathlib import Path
from datasets import load_dataset

# Your exact absolute paths
OUTPUT_DIR = Path("/Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs")
OUTPUT_FILE = OUTPUT_DIR / "captioning_train.jsonl"

# Create a new cache folder for RSICD images
IMAGE_CACHE_DIR = Path("/Users/farhanwajid/Web Development/SIH/downloaded_benchmarks/captioning_cache")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("📥 PIVOT: Streaming RSICD (Captioning) dataset from Hugging Face...")
print("🛡️ Bypassing the corrupted VRSBench repo to fulfill the SIH Captioning requirement.")

try:
    # Using the highly stable Remote Sensing Image Captioning Dataset (RSICD)
    dataset = load_dataset("arampacha/rsicd", split="train", streaming=True)
    
    samples = []
    MAX_SAMPLES = 2000  

    for idx, item in enumerate(dataset):
        if idx >= MAX_SAMPLES:
            break

        # Save image locally
        img_path = IMAGE_CACHE_DIR / f"rsicd_{idx}.png"
        if not img_path.exists() and "image" in item:
            item["image"].save(img_path)

        # RSICD provides multiple captions per image; we will just use the first one
        captions = item.get("captions", ["A satellite view of the earth."])
        caption = captions[0] if isinstance(captions, list) else captions

        # Format into Qwen2-VL Schema
        entry = {
            "id": f"captioning_{idx}",
            "images": [str(img_path.resolve())],
            "conversations": [
                {
                    "role": "user",
                    "content": f"<|vision_start|><|image_pad|><|vision_end|>\nProvide a detailed scene description of this satellite image."
                },
                {
                    "role": "assistant",
                    "content": str(caption)
                }
            ]
        }
        samples.append(entry)
        
        if (idx + 1) % 500 == 0:
            print(f"  • Processed {idx + 1} / {MAX_SAMPLES} samples...")

    # Save to JSONL
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for entry in samples:
            f.write(json.dumps(entry) + "\n")

    print(f"\n✅ Prepared {len(samples)} Captioning instruction pairs at: {OUTPUT_FILE}")

except Exception as e:
    print(f"⚠️ Error occurred: {e}")

/opt/anaconda3/envs/satquery-ft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 PIVOT: Streaming RSICD (Captioning) dataset from Hugging Face...
🛡️ Bypassing the corrupted VRSBench repo to fulfill the SIH Captioning requirement.


  • Processed 500 / 2000 samples...
  • Processed 1000 / 2000 samples...
  • Processed 1500 / 2000 samples...
  • Processed 2000 / 2000 samples...

✅ Prepared 2000 Captioning instruction pairs at: /Users/farhanwajid/Web Development/SIH/SatQuery-AI---An-Interactive-Vision-Language-Assistant-for-Multimodal-Remote-Sensing-Image-Analysis-/FineTuning-VLM-LLM/model_training/data_prep/Outputs/captioning_train.jsonl
